# Template análise de texto

In [ ]:
#### Passo 0: Instalar o Pandas e o OpenPyXL
#!pip install pandas openpyxl

In [ ]:
import os
import pandas as pd
import numpy as np
import random
import nltk
import matplotlib.pyplot as plt
import base64
from io import BytesIO
from nltk.corpus import stopwords
from wordcloud import WordCloud
from nltk.tokenize import word_tokenize
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer
from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import dendrogram, linkage
import re

# Download de recursos do NLTK
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('vader_lexicon')

# Fixando a seed para a aleatorização
np.random.seed(42)
random.seed(42)

# Mudar o diretório de trabalho
os.chdir('C://Users//diego.geronimo//experimentos_python_c//analise_texto_excel')

# Função para carregar o léxico
def carregar_op_lexicon_teste(caminho_arquivo):
    lexico = {}
    with open(caminho_arquivo, 'r', encoding='utf-8') as arquivo:
        for linha in arquivo:
            partes = linha.strip().split(',')
            if len(partes) >= 3:
                palavra, _, polaridade, _ = partes
                lexico[palavra] = polaridade
    return lexico

op_lexicon = carregar_op_lexicon_teste('lexico_v3.0.txt')

# Carregar dados
caminho_do_arquivo = 'base.xlsx'
df = pd.read_excel(caminho_do_arquivo, engine='openpyxl')

# Definir stopwords
stop_words = set(stopwords.words('portuguese')) | {'pro', 'pra', 'vi', 'muita','dia','ponto', 'pontos','vejo','ainda',
                                                   'acredito','pode','ter','fazer','todo','forma','sobre','todos',
                                                  'acho','cada','aqui','will','além','disso','alguns','sendo'}

# Função para remover stopwords e caracteres não alfabéticos
def clean_text(text):
    if not isinstance(text, str):
        return ""  # Retorna uma string vazia se não for texto
    texto = re.sub(r'[^\w\s]', '', text, flags=re.UNICODE)
    palavras = word_tokenize(texto.lower())
    return " ".join([palavra for palavra in palavras if palavra not in stop_words and palavra.isalpha()])

df['Texto'] = df['Texto'].apply(clean_text)

# Função para codificar imagens em base64
def image_to_base64(plt_figure):
    buf = BytesIO()
    plt_figure.savefig(buf, format='png')
    buf.seek(0)
    image_base64 = base64.b64encode(buf.getvalue()).decode('utf-8')
    buf.close()
    return image_base64

# Função para análise de sentimentos usando o OpLexicon
def analisar_sentimento_op_lexicon(texto, op_lexicon):
    palavras = word_tokenize(texto.lower())
    score = 0
    for palavra in palavras:
        if palavra in op_lexicon:
            score += int(op_lexicon[palavra])
        else:
            score += 0
    return score


# Função para salvar a nuvem de palavras e retornar como base64
def save_wordcloud(texto):
    palavras = texto.split()
    contagem_palavras = Counter(palavras)
    palavras_top_30 = dict(contagem_palavras.most_common(30))
    wordcloud = WordCloud(width=800, height=400, background_color='white').generate_from_frequencies(palavras_top_30)
    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    return image_to_base64(plt)

# Função para análise hierárquica e salvar dendrograma como base64
def save_dendrogram(textos):
    vectorizer = CountVectorizer()
    X = vectorizer.fit_transform(textos)
    terms = vectorizer.get_feature_names_out()
    term_freq = np.asarray(X.sum(axis=0)).ravel()
    frequent_terms = [term for term, freq in sorted(zip(terms, term_freq), key=lambda x: x[1], reverse=True)[:30]]
    vectorizer = CountVectorizer(vocabulary=frequent_terms)
    X_top = vectorizer.fit_transform(textos)
    X_bin_top = X_top.toarray()
    distance_matrix = pdist(X_bin_top.T, 'jaccard')
    linkage_matrix = linkage(distance_matrix, 'average')
    plt.figure(figsize=(10, 5))
    dendrogram(linkage_matrix, labels=frequent_terms, leaf_rotation=90, leaf_font_size=12)
    plt.title("Dendrogram")
    plt.xlabel("Terms")
    plt.ylabel("Distance")
    plt.tight_layout()
    return image_to_base64(plt)

# Análise de sentimentos
df['sentiment_score'] = df['Texto'].apply(analisar_sentimento_op_lexicon, args=(op_lexicon,))
#media_score = df['sentiment_score'].mean()
media_score = round(df['sentiment_score'].mean(), 2)

# Geração de imagens como base64
wordcloud_image = save_wordcloud(' '.join(df['Texto'].tolist()))
dendrogram_image = save_dendrogram(df['Texto'].tolist())

# Seleção aleatória de textos
random_responses = df['Texto'].sample(n=min(5, len(df)), replace=False).tolist()

# Construindo e salvando o HTML
html_content = f"""
<html>
<head><title>Relatório de Análise</title></head>
<body>
<h1>Média de Score de Sentimento: {media_score}</h1>
<h1>Nuvem de Palavras</h1>
<img src="data:image/png;base64,{wordcloud_image}" alt="Wordcloud">
<h1>Seleção Aleatória de Respostas</h1>
<p>{'<br>'.join(random_responses)}</p>
<h1>Dendrograma de Clustering Hierárquico</h1>
<img src="data:image/png;base64,{dendrogram_image}" alt="Dendrogram">
</body>
</html>
"""

with open("report.html", "w") as file:
    file.write(html_content)
